# Fase 2 — Fine-Tuning, Redução de Tamanho e Estabilização do Detector de Segmentação (`RF-DETR-Seg`)

Este notebook executa no **Google Colab Pro (GPU T4 ou L4)** toda a **Fase 2 de Otimização do Detector de Segmentação de Tabuleiro (`RF-DETR-Seg`)**, focado em:
1. **Economia de créditos no Colab Pro:** cópia e descompactação do dataset COCO do Google Drive para o SSD local da máquina virtual (`/content/dataset_coco/`) antes do treino, eliminando gargalo de I/O de rede, e persistência imediata de todos os resultados no Google Drive (`PASTA_SAIDA_DRIVE`).
2. **Saneamento rigoroso das anotações COCO:** unificação em exatamente **1 categoria alvo (`tabuleiro`)** e remoção de polígonos degenerados.
3. **Augmentation Geométrico e Fotométrico Direcionado aos 4 Cantos:** variação de escala multi-distância (`0.65x` a `1.15x`), inclinação projetiva leve (`0.02` a `0.06`), rotações ortogonais (`0°, 90°, 180°, 270°`), espelhamento horizontal, temperatura de cor/brilho/contraste (`±15%`) e sombras suaves de mesa, com **validação geométrica pós-transformação** garantindo que os 4 vértices do tabuleiro nunca sejam amputados.
4. **Estudo Optuna Persistente em SQLite no Drive (`15 trials`):** busca sobre variantes compactas (`RFDETRSegNano`, `RFDETRSegSmall`, `RFDETRSegMedium`), resoluções (`560`, `616`), `lr`, `weight_decay` e `grad_accum_steps`, maximizando:
   $$\text{Score}_{\text{Fase2}} = 0.45 \times \text{IoU}_{\text{Medio\_Mascara}} + 0.35 \times \text{mAP}_{50:95\text{\_Mascara}} + 0.20 \times \text{Taxa}_{\text{Cantos\_Precisos\_10px}}$$
5. **Exportação ONNX Limpa (`opset 17`, `dynamo=False`) e Relatório Visual Colorido (`300 dpi`):** verificação de tamanho (`meta <= 45 MB`), ausência de `NaN`/`Inf` em 20 imagens reais, paridade PyTorch vs. ONNX Runtime (`erro_max < 1e-3`), 4 gráficos coloridos em alta resolução, `relatorio_fase2.json` e célula final autônoma de reexportação ONNX em CPU.

## 1. Montagem do Drive, Cópia da Entrada para SSD Local e Verificação de GPU
- Monta o Google Drive em `/content/drive`.
- Configura os caminhos oficiais da Fase 2 e cria `PASTA_SAIDA_DRIVE` automaticamente.
- Instala as dependências (`rfdetr`, `supervision`, `optuna`, `onnx`, `onnxruntime`, `opencv-python-headless`, `seaborn`, `pycocotools`).
- Copia `ARQUIVO_ZIP_COCO` para o SSD local (`/content/`), extrai em `/content/dataset_coco/` e saneia `_annotations.coco.json` em `train/`, `valid/` e `test/`.

In [ ]:
!pip install -q "rfdetr[train]" supervision optuna onnx onnxruntime opencv-python-headless seaborn pycocotools scikit-learn matplotlib

In [ ]:
import copy
import gc
import glob
import json
import math
import os
import random
import shutil
import sys
import time
import warnings
import zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import onnx
import onnxruntime as ort
import optuna
import seaborn as sns
import torch
from PIL import Image

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.INFO)

if "google.colab" in sys.modules:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

ARQUIVO_ZIP_COCO = "/content/drive/MyDrive/doutorado/20261 visao computacional/projeto/dataset2/kingdomino_pontuador.v2-v2.coco-segmentation.zip"
PASTA_ORIGINAIS_DRIVE = "/content/drive/MyDrive/doutorado/20261 visao computacional/projeto/dataset2/lote_1_e_2"
PASTA_PROJETO_FASE2 = "/content/drive/MyDrive/doutorado/20261 visao computacional/projeto/dataset3"
PASTA_SAIDA_DRIVE = f"{PASTA_PROJETO_FASE2}/resultados_fase2"

os.makedirs(PASTA_SAIDA_DRIVE, exist_ok=True)

PASTA_SSD_LOCAL = "/content"
CAMINHO_ZIP_LOCAL = os.path.join(PASTA_SSD_LOCAL, "kingdomino_coco_v2.zip")
PASTA_DATASET_COCO_BASE = os.path.join(PASTA_SSD_LOCAL, "dataset_coco_base")
PASTA_DATASET_COCO = os.path.join(PASTA_SSD_LOCAL, "dataset_coco")

SEMENTE_GLOBAL = 42


def fixar_sementes(semente=42):
    random.seed(semente)
    np.random.seed(semente)
    torch.manual_seed(semente)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(semente)
        torch.backends.cudnn.benchmark = True


fixar_sementes(SEMENTE_GLOBAL)

TEM_GPU = torch.cuda.is_available()
DISPOSITIVO = torch.device("cuda" if TEM_GPU else "cpu")

print("=" * 82)
print("CONFIGURAÇÃO DO AMBIENTE — FASE 2 (DETECTOR RF-DETR-SEG)")
print("=" * 82)
if TEM_GPU:
    nome_gpu = torch.cuda.get_device_name(0)
    vram_total_gb = torch.cuda.get_device_properties(0).total_memory / (1024.0 ** 3)
    vram_livre_gb = torch.cuda.mem_get_info()[0] / (1024.0 ** 3)
    print(f"GPU Ativa:                {nome_gpu}")
    print(f"VRAM Total / Livre:       {vram_total_gb:.2f} GB / {vram_livre_gb:.2f} GB")
else:
    print("AVISO: GPU não detectada! Em produção no Colab Pro selecione GPU T4 ou L4.")
print(f"Arquivo ZIP COCO (Drive): {ARQUIVO_ZIP_COCO}")
print(f"Pasta Originais (Drive):  {PASTA_ORIGINAIS_DRIVE}")
print(f"Pasta Saída (Drive):      {PASTA_SAIDA_DRIVE}")
print(f"Dataset no SSD Local:     {PASTA_DATASET_COCO}")
print("=" * 82)

In [ ]:
def calcular_area_poligono(pontos_xy):
    pts = np.asarray(pontos_xy, dtype=np.float32).reshape(-1, 2)
    if len(pts) < 3:
        return 0.0
    return float(cv2.contourArea(pts))


def sanear_arquivo_coco_json(caminho_json, pasta_split):
    """
    Garante que o arquivo _annotations.coco.json possua exatamente 1 categoria alvo
    ('tabuleiro', id=0 ou id=1 consistente sem supercategoria fantasma), removendo
    polígonos degenerados (< 3 vértices ou área < 25 px²) e imagens inexistentes.
    """
    with open(caminho_json, "r", encoding="utf-8") as arq:
        dados = json.load(arq)

    imagens_validas = []
    mapa_imgs = {}
    for img in dados.get("images", []):
        caminho_img = os.path.join(pasta_split, img["file_name"])
        if os.path.isfile(caminho_img):
            imagens_validas.append(img)
            mapa_imgs[int(img["id"])] = img

    anotacoes_saneadas = []
    novo_id_anot = 1
    descartadas = 0

    for anot in dados.get("annotations", []):
        img_id = int(anot.get("image_id", -1))
        if img_id not in mapa_imgs:
            descartadas += 1
            continue

        info_img = mapa_imgs[img_id]
        larg_img = float(info_img.get("width", 640))
        alt_img = float(info_img.get("height", 640))

        segs_brutas = anot.get("segmentation", [])
        if not isinstance(segs_brutas, list) or len(segs_brutas) == 0:
            descartadas += 1
            continue

        segs_limpas = []
        todos_xs = []
        todos_ys = []
        area_total = 0.0

        for pol in segs_brutas:
            if not isinstance(pol, list) or len(pol) < 6 or len(pol) % 2 != 0:
                continue
            pts = np.asarray(pol, dtype=np.float32).reshape(-1, 2)
            pts[:, 0] = np.clip(pts[:, 0], 0.0, larg_img - 1.0)
            pts[:, 1] = np.clip(pts[:, 1], 0.0, alt_img - 1.0)
            area_pol = calcular_area_poligono(pts)
            if area_pol < 25.0:
                continue
            segs_limpas.append([round(float(v), 2) for v in pts.flatten()])
            todos_xs.extend(pts[:, 0].tolist())
            todos_ys.extend(pts[:, 1].tolist())
            area_total += area_pol

        if not segs_limpas:
            descartadas += 1
            continue

        x_min = float(min(todos_xs))
        y_min = float(min(todos_ys))
        x_max = float(max(todos_xs))
        y_max = float(max(todos_ys))
        larg_box = max(2.0, x_max - x_min)
        alt_box = max(2.0, y_max - y_min)

        anotacoes_saneadas.append({
            "id": novo_id_anot,
            "image_id": img_id,
            "category_id": 0,
            "bbox": [
                round(x_min, 2),
                round(y_min, 2),
                round(larg_box, 2),
                round(alt_box, 2),
            ],
            "area": round(area_total, 2),
            "segmentation": segs_limpas,
            "iscrowd": 0,
        })
        novo_id_anot += 1

    # Mantém apenas imagens que possuem ao menos 1 anotação válida de tabuleiro
    imgs_com_anot = {a["image_id"] for a in anotacoes_saneadas}
    imagens_finais = [img for img in imagens_validas if int(img["id"]) in imgs_com_anot]

    dados_saneados = {
        "info": dados.get("info", {"description": "Kingdomino Tabuleiro COCO Saneado"}),
        "licenses": dados.get("licenses", []),
        "categories": [
            {"id": 0, "name": "tabuleiro", "supercategory": "none"}
        ],
        "images": imagens_finais,
        "annotations": anotacoes_saneadas,
    }

    with open(caminho_json, "w", encoding="utf-8") as arq:
        json.dump(dados_saneados, arq, ensure_ascii=False, indent=2)

    return {
        "imagens": len(imagens_finais),
        "anotacoes": len(anotacoes_saneadas),
        "descartadas": descartadas,
    }


def copiar_e_preparar_dataset_no_ssd(
    caminho_zip_drive,
    caminho_zip_local,
    pasta_base_ssd,
    pasta_trabalho_ssd,
):
    for pasta in (pasta_base_ssd, pasta_trabalho_ssd):
        if os.path.exists(pasta):
            shutil.rmtree(pasta)

    if os.path.isfile(caminho_zip_drive):
        print(f"Copiando ZIP do Google Drive para o SSD local: {caminho_zip_local} ...")
        shutil.copy2(caminho_zip_drive, caminho_zip_local)
        with zipfile.ZipFile(caminho_zip_local, "r") as zip_ref:
            zip_ref.extractall(pasta_base_ssd)
    elif os.path.isdir(os.path.join("dados", "tabuleiro_coco")):
        print("Usando dataset COCO local de fallback (dados/tabuleiro_coco)...")
        shutil.copytree(os.path.join("dados", "tabuleiro_coco"), pasta_base_ssd)
    else:
        raise FileNotFoundError(
            f"Arquivo ZIP COCO não encontrado no Drive: {caminho_zip_drive}"
        )

    # Caso o ZIP tenha sido extraído dentro de uma subpasta única, nivela para pasta_base_ssd
    if not os.path.isdir(os.path.join(pasta_base_ssd, "train")):
        for raiz_atual, subpastas, _ in os.walk(pasta_base_ssd):
            if "train" in subpastas:
                for sub in ("train", "valid", "test"):
                    orig = os.path.join(raiz_atual, sub)
                    dest = os.path.join(pasta_base_ssd, sub)
                    if os.path.isdir(orig) and orig != dest:
                        shutil.move(orig, dest)
                break

    # Se não existir split 'test', cria a partir de uma cópia de 'valid'
    pasta_val_base = os.path.join(pasta_base_ssd, "valid")
    pasta_test_base = os.path.join(pasta_base_ssd, "test")
    if not os.path.isdir(pasta_test_base) and os.path.isdir(pasta_val_base):
        shutil.copytree(pasta_val_base, pasta_test_base)

    resumo_splits = {}
    for split in ("train", "valid", "test"):
        pasta_s = os.path.join(pasta_base_ssd, split)
        caminho_j = os.path.join(pasta_s, "_annotations.coco.json")
        if os.path.isfile(caminho_j):
            resumo_splits[split] = sanear_arquivo_coco_json(caminho_j, pasta_s)

    shutil.copytree(pasta_base_ssd, pasta_trabalho_ssd)
    return resumo_splits


RESUMO_COCO_INICIAL = copiar_e_preparar_dataset_no_ssd(
    ARQUIVO_ZIP_COCO,
    CAMINHO_ZIP_LOCAL,
    PASTA_DATASET_COCO_BASE,
    PASTA_DATASET_COCO,
)

print("\nRESUMO DO DATASET COCO SANEADO NO SSD LOCAL (/content/dataset_coco):")
for nome_split, info_s in RESUMO_COCO_INICIAL.items():
    print(
        f"  - {nome_split:<6}: {info_s['imagens']:3d} imagens | "
        f"{info_s['anotacoes']:3d} anotações de 'tabuleiro' | "
        f"{info_s['descartadas']:2d} degeneradas removidas"
    )

## 2. Pipeline de Augmentation Geométrico e Fotométrico Direcionado aos 4 Cantos
Constrói o gerador de transformações exclusivamente sobre o conjunto `train`, com **validação geométrica pós-transformação (`validar_preservacao_4_cantos`)** que rejeita qualquer transformação que corte ou ampute algum dos 4 vértices do tabuleiro:
- **Escala multi-distância (`0.65x` a `1.15x`):** simula fotos capturadas de perto (tabuleiros `7x7`) e mais afastadas (`5x5`).
- **Inclinação projetiva leve (`0.02` a `0.06`), rotações ortogonais (`0°, 90°, 180°, 270°`) e espelhamento horizontal.**
- **Variação fotométrica (`±15%` em brilho, contraste e temperatura de cor) + sombras suaves de mesa:** evita vazamento da máscara para peças soltas ou texturas da superfície.

In [ ]:
def ordenar_vertices_horario_tl_tr_br_bl(pontos):
    """
    Ordena 4 vértices no sentido horário iniciando pelo canto superior esquerdo
    (1: TL, 2: TR, 3: BR, 4: BL) usando ângulo polar em relação ao centroide.
    """
    pts = np.asarray(pontos, dtype=np.float32).reshape(-1, 2)
    centro = np.mean(pts, axis=0)
    angulos = np.arctan2(pts[:, 1] - centro[1], pts[:, 0] - centro[0])
    ordem = np.argsort(angulos)
    pts_ord = pts[ordem]
    idx_tl = int(np.argmin(pts_ord[:, 0] + pts_ord[:, 1]))
    return np.roll(pts_ord, -idx_tl, axis=0)


def extrair_4_cantos_de_mascara_ou_contorno(mascara_ou_contorno):
    """
    Extrai os 4 cantos ordenados (TL, TR, BR, BL) a partir de uma máscara binária
    ou contorno poligonal usando cv2.findContours + busca binária em cv2.approxPolyDP
    e poda iterativa de menor área triangular.
    """
    if mascara_ou_contorno.ndim == 2 and mascara_ou_contorno.shape[1] != 2:
        mascara_u8 = (mascara_ou_contorno > 0).astype(np.uint8) * 255
        contornos, _ = cv2.findContours(
            mascara_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        if not contornos:
            return None
        contorno = max(contornos, key=cv2.contourArea)
    else:
        contorno = np.asarray(mascara_ou_contorno, dtype=np.float32).reshape(-1, 1, 2)

    envolvente = cv2.convexHull(contorno.astype(np.float32))
    perimetro = cv2.arcLength(envolvente, True)
    if perimetro <= 1e-3:
        return None

    baixo = 0.001
    alto = 0.20
    melhor_quad = None
    for _ in range(30):
        meio = 0.5 * (baixo + alto)
        aprox = cv2.approxPolyDP(envolvente, meio * perimetro, True)
        if len(aprox) == 4:
            melhor_quad = aprox.reshape(4, 2)
            break
        elif len(aprox) > 4:
            baixo = meio
        else:
            alto = meio

    if melhor_quad is not None:
        return ordenar_vertices_horario_tl_tr_br_bl(melhor_quad)

    pts = envolvente.reshape(-1, 2).astype(np.float32)
    while len(pts) > 4:
        areas = []
        n = len(pts)
        for i in range(n):
            p_ant = pts[(i - 1) % n]
            p_at = pts[i]
            p_pr = pts[(i + 1) % n]
            area_t = 0.5 * abs(
                (p_at[0] - p_ant[0]) * (p_pr[1] - p_ant[1])
                - (p_at[1] - p_ant[1]) * (p_pr[0] - p_ant[0])
            )
            areas.append(area_t)
        pts = np.delete(pts, int(np.argmin(areas)), axis=0)

    if len(pts) == 4:
        return ordenar_vertices_horario_tl_tr_br_bl(pts)

    ret = cv2.minAreaRect(envolvente)
    return ordenar_vertices_horario_tl_tr_br_bl(cv2.boxPoints(ret))


def validar_preservacao_4_cantos(lista_poligonos, largura, altura, margem_borda_px=4.0):
    """
    Verifica se todos os vértices dos polígonos transformados permanecem estritamente
    dentro do quadro [margem_borda_px, largura - margem_borda_px] x [margem_borda_px, altura - margem_borda_px]
    e se a área e convexidade dos 4 cantos foram preservadas sem amputação.
    """
    if not lista_poligonos:
        return False

    for pts in lista_poligonos:
        pts_arr = np.asarray(pts, dtype=np.float32).reshape(-1, 2)
        if len(pts_arr) < 4:
            return False
        if (
            np.any(pts_arr[:, 0] < margem_borda_px)
            or np.any(pts_arr[:, 0] > (largura - 1.0 - margem_borda_px))
            or np.any(pts_arr[:, 1] < margem_borda_px)
            or np.any(pts_arr[:, 1] > (altura - 1.0 - margem_borda_px))
        ):
            return False

        area_p = calcular_area_poligono(pts_arr)
        if area_p < (largura * altura * 0.03):
            return False

        cantos_4 = extrair_4_cantos_de_mascara_ou_contorno(pts_arr)
        if cantos_4 is None or len(cantos_4) != 4:
            return False
        lados = [np.linalg.norm(cantos_4[(i + 1) % 4] - cantos_4[i]) for i in range(4)]
        if min(lados) < 25.0:
            return False

    return True

In [ ]:
def aplicar_fotometria_e_sombra_suave(imagem_bgr):
    """
    Aplica variação de brilho, contraste e temperatura de cor (+-15%) e projeta
    uma sombra linear/elíptica suave sobre a mesa para robustez contra iluminação lateral.
    """
    img = imagem_bgr.astype(np.float32)

    # Brilho e contraste (+-15%)
    fator_contraste = random.uniform(0.85, 1.15)
    delta_brilho = random.uniform(-0.15, 0.15) * 255.0
    media_atual = float(np.mean(img))
    img = (img - media_atual) * fator_contraste + media_atual + delta_brilho

    # Temperatura de cor (+-15% balanço R/B)
    fator_temp = random.uniform(-0.12, 0.12)
    img[:, :, 2] = img[:, :, 2] * (1.0 + fator_temp)
    img[:, :, 0] = img[:, :, 0] * (1.0 - fator_temp)

    # Sombra suave em 50% das amostras
    if random.random() < 0.50:
        alt, larg = img.shape[:2]
        yy, xx = np.mgrid[0:alt, 0:larg].astype(np.float32)
        ang = random.uniform(0, 2 * math.pi)
        proj = (xx - larg * 0.5) * math.cos(ang) + (yy - alt * 0.5) * math.sin(ang)
        largura_transicao = max(40.0, min(larg, alt) * random.uniform(0.18, 0.35))
        desloc = random.uniform(-0.25, 0.25) * min(larg, alt)
        mascara_sombra = 1.0 / (1.0 + np.exp(-(proj - desloc) / largura_transicao))
        intensidade_sombra = random.uniform(0.15, 0.32)
        fator_luz = 1.0 - intensidade_sombra * mascara_sombra[..., np.newaxis]
        img = img * fator_luz

    return np.clip(img, 0, 255).astype(np.uint8)


def transformar_imagem_e_poligonos_preservando_cantos(
    imagem_bgr,
    lista_poligonos,
    max_tentativas=15,
):
    """
    Aplica escala multi-distância (0.65x a 1.15x), inclinação projetiva leve (0.02 a 0.06),
    rotação ortogonal (0, 90, 180, 270) e espelhamento horizontal, validando que os 4 cantos
    do polígono do tabuleiro nunca sejam amputados.
    """
    alt, larg = imagem_bgr.shape[:2]
    centro = np.array([larg * 0.5, alt * 0.5], dtype=np.float32)

    for _ in range(max_tentativas):
        escala = random.uniform(0.65, 1.15)
        angulo_ortogonal = random.choice((0, 90, 180, 270))
        espelhar_h = random.random() < 0.50
        intensidade_persp = random.uniform(0.02, 0.06)

        # Constrói homografia combinada centrada na imagem
        rad = math.radians(angulo_ortogonal)
        cos_a = math.cos(rad) * escala
        sin_a = math.sin(rad) * escala

        mat_rot_esc = np.array([
            [cos_a, -sin_a, centro[0] * (1.0 - cos_a) + centro[1] * sin_a],
            [sin_a,  cos_a, centro[1] * (1.0 - cos_a) - centro[0] * sin_a],
            [0.0,    0.0,   1.0],
        ], dtype=np.float32)

        if espelhar_h:
            mat_flip = np.array([
                [-1.0, 0.0, larg - 1.0],
                [0.0,  1.0, 0.0],
                [0.0,  0.0, 1.0],
            ], dtype=np.float32)
            mat_rot_esc = mat_flip @ mat_rot_esc

        cantos_img = np.array([
            [0.0, 0.0],
            [larg - 1.0, 0.0],
            [larg - 1.0, alt - 1.0],
            [0.0, alt - 1.0],
        ], dtype=np.float32)
        desvios = np.random.uniform(
            -intensidade_persp, intensidade_persp, size=(4, 2)
        ).astype(np.float32) * np.array([larg, alt], dtype=np.float32)
        mat_persp = cv2.getPerspectiveTransform(cantos_img, cantos_img + desvios).astype(np.float32)

        mat_h = mat_persp @ mat_rot_esc

        poligonos_transf = []
        for pol in lista_poligonos:
            pts = np.asarray(pol, dtype=np.float32).reshape(-1, 1, 2)
            pts_t = cv2.perspectiveTransform(pts, mat_h).reshape(-1, 2)
            poligonos_transf.append(pts_t)

        if validar_preservacao_4_cantos(poligonos_transf, larg, alt, margem_borda_px=5.0):
            cor_borda = tuple(int(v) for v in np.median(imagem_bgr, axis=(0, 1)))
            img_warp = cv2.warpPerspective(
                imagem_bgr,
                mat_h,
                (larg, alt),
                flags=cv2.INTER_LINEAR,
                borderMode=cv2.BORDER_CONSTANT,
                borderValue=cor_borda,
            )
            img_final = aplicar_fotometria_e_sombra_suave(img_warp)
            return img_final, poligonos_transf, True

    # Fallback seguro: apenas variação fotométrica e rotação ortogonal se o polígono já estiver na borda extrema
    return aplicar_fotometria_e_sombra_suave(imagem_bgr), [np.asarray(p, dtype=np.float32).reshape(-1, 2) for p in lista_poligonos], False


def regenerar_split_treino_aumentado_ssd(pasta_base_ssd, pasta_trabalho_ssd, semente_epoca=42):
    """
    Restaura as imagens originais de treino a partir de pasta_base_ssd e gera um conjunto
    de treino com augmentations geométricos e fotométricos validados (4 cantos intactos).
    """
    fixar_sementes(semente_epoca)
    pasta_train_base = os.path.join(pasta_base_ssd, "train")
    pasta_train_dest = os.path.join(pasta_trabalho_ssd, "train")

    if os.path.exists(pasta_train_dest):
        shutil.rmtree(pasta_train_dest)
    shutil.copytree(pasta_train_base, pasta_train_dest)

    caminho_json = os.path.join(pasta_train_dest, "_annotations.coco.json")
    with open(caminho_json, "r", encoding="utf-8") as arq:
        dados = json.load(arq)

    anots_por_img = {}
    for anot in dados.get("annotations", []):
        anots_por_img.setdefault(int(anot["image_id"]), []).append(anot)

    novas_anotacoes = []
    id_anot = 1
    qtd_geom_aplicada = 0

    for info_img in dados.get("images", []):
        img_id = int(info_img["id"])
        lista_a = anots_por_img.get(img_id, [])
        if not lista_a:
            continue

        caminho_img = os.path.join(pasta_train_dest, info_img["file_name"])
        img_bgr = cv2.imread(caminho_img)
        if img_bgr is None:
            continue

        # Aplica augmentation geométrico+fotométrico em 75% das imagens de treino
        if random.random() < 0.75:
            pols_orig = [np.asarray(a["segmentation"][0], dtype=np.float32).reshape(-1, 2) for a in lista_a]
            img_aug, pols_novos, sucesso_geom = transformar_imagem_e_poligonos_preservando_cantos(
                img_bgr, pols_orig
            )
            if sucesso_geom:
                qtd_geom_aplicada += 1
            cv2.imwrite(caminho_img, img_aug)
        else:
            pols_novos = [np.asarray(a["segmentation"][0], dtype=np.float32).reshape(-1, 2) for a in lista_a]

        for anot_orig, pts_novo in zip(lista_a, pols_novos):
            x_min = float(np.min(pts_novo[:, 0]))
            y_min = float(np.min(pts_novo[:, 1]))
            x_max = float(np.max(pts_novo[:, 0]))
            y_max = float(np.max(pts_novo[:, 1]))
            area_n = calcular_area_poligono(pts_novo)

            novas_anotacoes.append({
                "id": id_anot,
                "image_id": img_id,
                "category_id": 0,
                "bbox": [
                    round(x_min, 2),
                    round(y_min, 2),
                    round(max(2.0, x_max - x_min), 2),
                    round(max(2.0, y_max - y_min), 2),
                ],
                "area": round(area_n, 2),
                "segmentation": [[round(float(v), 2) for v in pts_novo.flatten()]],
                "iscrowd": 0,
            })
            id_anot += 1

    dados["annotations"] = novas_anotacoes
    with open(caminho_json, "w", encoding="utf-8") as arq:
        json.dump(dados, arq, ensure_ascii=False, indent=2)

    return len(dados.get("images", [])), qtd_geom_aplicada

In [ ]:
# Auditoria Visual do Augmentation Geométrico e Fotométrico Preservando os 4 Cantos
def visualizar_auditoria_augmentation_4_cantos(pasta_base_ssd):
    pasta_train = os.path.join(pasta_base_ssd, "train")
    caminho_json = os.path.join(pasta_train, "_annotations.coco.json")
    with open(caminho_json, "r", encoding="utf-8") as arq:
        dados = json.load(arq)

    anots_por_img = {}
    for a in dados["annotations"]:
        anots_por_img.setdefault(int(a["image_id"]), []).append(a)

    amostras = random.sample(dados["images"], min(3, len(dados["images"])))
    cores_cantos = [(255, 50, 50), (50, 220, 50), (50, 120, 255), (255, 210, 0)]
    nomes_cantos = ["1:TL", "2:TR", "3:BR", "4:BL"]

    fig, eixos = plt.subplots(2, 3, figsize=(13.5, 8.5), dpi=130)
    fig.suptitle(
        "Auditoria do Augmentation Geométrico/Fotométrico com Preservação dos 4 Cantos (TL, TR, BR, BL)",
        fontsize=12.5,
        fontweight="bold",
    )

    for col, info_img in enumerate(amostras):
        img_bgr = cv2.imread(os.path.join(pasta_train, info_img["file_name"]))
        pols = [np.asarray(a["segmentation"][0], dtype=np.float32).reshape(-1, 2) for a in anots_por_img[int(info_img["id"])]]
        img_aug, pols_aug, _ = transformar_imagem_e_poligonos_preservando_cantos(img_bgr, pols)

        for lin, (im_atual, pl_atual, tit) in enumerate([
            (img_bgr, pols, "Original (Treino)"),
            (img_aug, pols_aug, "Augmentado (4 Cantos Preservados)"),
        ]):
            vis = cv2.cvtColor(im_atual.copy(), cv2.COLOR_BGR2RGB)
            for p in pl_atual:
                pts_i = np.round(p).astype(np.int32)
                cv2.polylines(vis, [pts_i], True, (0, 255, 255), 2)
                cantos = extrair_4_cantos_de_mascara_ou_contorno(p)
                if cantos is not None:
                    for idx_c, (cx, cy) in enumerate(cantos):
                        cv2.circle(vis, (int(cx), int(cy)), 7, cores_cantos[idx_c], -1)
                        cv2.putText(
                            vis,
                            nomes_cantos[idx_c],
                            (int(cx) + 6, int(cy) - 6),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.48,
                            cores_cantos[idx_c],
                            2,
                            cv2.LINE_AA,
                        )
            eixos[lin, col].imshow(vis)
            eixos[lin, col].set_title(tit, fontsize=10, fontweight="bold")
            eixos[lin, col].axis("off")

    plt.tight_layout()
    plt.show()


visualizar_auditoria_augmentation_4_cantos(PASTA_DATASET_COCO_BASE)

## 3. Estudo Automatizado com Optuna Persistente em SQLite no Drive (15 Trials)
- **Persistência no Drive:** `sqlite:///{PASTA_SAIDA_DRIVE}/estudo_optuna_fase2.db` com `load_if_exists=True` (permite retomar exatamente do último trial concluído se a sessão do Colab cair).
- **Métrica Objetivo (`Score_Fase2`):**
  $$\text{Score}_{\text{Fase2}} = 0.45 \times \text{IoU}_{\text{Medio\_Mascara}} + 0.35 \times \text{mAP}_{50:95\text{\_Mascara}} + 0.20 \times \text{Taxa}_{\text{Cantos\_Precisos\_10px}}$$
- **Espaço de Busca:**
  - Variante compacta: `RFDETRSegNano`, `RFDETRSegSmall` ou `RFDETRSegMedium`;
  - Resolução de entrada múltipla de `56` (`560` ou `616`, com ajuste automático para o divisor de janela do backbone quando exigido pela versão instalada do `rfdetr`);
  - Taxa de aprendizado (`5e-5` a `3e-4`), `weight_decay` (`1e-4` a `1e-2`) e `grad_accum_steps` calibrado para a VRAM da GPU T4/L4.

In [ ]:
import rfdetr

MAPA_CLASSES_RFDETR = {}
for nome_var in ("RFDETRSegNano", "RFDETRSegSmall", "RFDETRSegMedium"):
    if hasattr(rfdetr, nome_var):
        MAPA_CLASSES_RFDETR[nome_var] = getattr(rfdetr, nome_var)

print("Variantes RF-DETR-Seg disponíveis no pacote:", list(MAPA_CLASSES_RFDETR.keys()))


def ajustar_resolucao_valida_rfdetr(classe_modelo, resolucao_solicitada):
    """
    Verifica o divisor exigido pelo backbone (patch_size * num_windows, ex: 12, 24 ou 56)
    e garante que a resolução seja aceita sem erro de divisibilidade.
    """
    divisor = 56
    cfg_cls = getattr(classe_modelo, "_model_config_class", None)
    if cfg_cls is not None:
        try:
            inst_cfg = cfg_cls()
            ps = int(getattr(inst_cfg, "patch_size", 12))
            nw = int(getattr(inst_cfg, "num_windows", 2))
            divisor = max(1, ps * nw)
        except Exception:
            divisor = 24

    if resolucao_solicitada % divisor == 0:
        return int(resolucao_solicitada)
    return int(round(resolucao_solicitada / float(divisor)) * divisor)


def instanciar_detector_rfdetr(nome_variante, resolucao_solicitada, caminho_pesos=None):
    classe_modelo = MAPA_CLASSES_RFDETR[nome_variante]
    resolucao_real = ajustar_resolucao_valida_rfdetr(classe_modelo, resolucao_solicitada)
    kwargs = {"resolution": resolucao_real, "num_classes": 1}
    if caminho_pesos is not None and os.path.isfile(caminho_pesos):
        kwargs["pretrain_weights"] = caminho_pesos
    modelo = classe_modelo(**kwargs)
    return modelo, resolucao_real

In [ ]:
def calcular_iou_mascaras_binarias(mascara_pred, mascara_gt):
    b_pred = mascara_pred > 0
    b_gt = mascara_gt > 0
    inter = float(np.logical_and(b_pred, b_gt).sum())
    uniao = float(np.logical_or(b_pred, b_gt).sum())
    return (inter / uniao) if uniao > 0 else 0.0


def avaliar_geometria_detector_no_split(modelo_rfdetr, pasta_split, resolucao_inferencia, limiar_confianca=0.35):
    """
    Avalia um detector RF-DETR-Seg em todas as imagens de um split COCO ('valid' ou 'test'),
    calculando:
      - IoU_Medio_Mascara
      - mAP50_95_Mascara (média nos limiares de IoU de 0.50 a 0.95 com passo 0.05)
      - Erro euclidiano dos 4 cantos (TL, TR, BR, BL) na resolução de inferência
      - Taxa_Cantos_Precisos_10px (% de tabuleiros com erro médio dos 4 cantos < 10 px)
      - Score_Fase2 = 0.45 * IoU_Medio_Mascara + 0.35 * mAP50_95_Mascara + 0.20 * Taxa_Cantos_Precisos_10px
    """
    caminho_json = os.path.join(pasta_split, "_annotations.coco.json")
    with open(caminho_json, "r", encoding="utf-8") as arq:
        dados = json.load(arq)

    anots_por_img = {}
    for a in dados.get("annotations", []):
        anots_por_img.setdefault(int(a["image_id"]), []).append(a)

    ious = []
    erros_cantos_px = []
    registros_por_imagem = []
    limiares_map = np.arange(0.50, 0.96, 0.05)

    for info_img in dados.get("images", []):
        img_id = int(info_img["id"])
        lista_a = anots_por_img.get(img_id, [])
        if not lista_a:
            continue

        caminho_img = os.path.join(pasta_split, info_img["file_name"])
        img_bgr = cv2.imread(caminho_img)
        if img_bgr is None:
            continue
        alt_orig, larg_orig = img_bgr.shape[:2]
        escala_x = float(resolucao_inferencia) / float(larg_orig)
        escala_y = float(resolucao_inferencia) / float(alt_orig)

        # Constrói máscara e 4 cantos ground-truth do polígono principal
        melhor_anot = max(lista_a, key=lambda x: float(x.get("area", 0.0)))
        pts_gt = np.asarray(melhor_anot["segmentation"][0], dtype=np.float32).reshape(-1, 2)
        mascara_gt = np.zeros((alt_orig, larg_orig), dtype=np.uint8)
        cv2.fillPoly(mascara_gt, [np.round(pts_gt).astype(np.int32)], 255)

        pts_gt_res = pts_gt.copy()
        pts_gt_res[:, 0] *= escala_x
        pts_gt_res[:, 1] *= escala_y
        cantos_gt_res = extrair_4_cantos_de_mascara_ou_contorno(pts_gt_res)

        img_rgb_pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        deteccoes = modelo_rfdetr.predict(img_rgb_pil, threshold=limiar_confianca)

        iou_melhor = 0.0
        erro_cantos_img = float(resolucao_inferencia)
        cantos_pred_orig = None
        mascara_pred_escolhida = None
        conf_escolhida = 0.0

        if deteccoes is not None and len(deteccoes) > 0 and deteccoes.mask is not None:
            for idx_d in range(len(deteccoes)):
                m_bruta = np.asarray(deteccoes.mask[idx_d], dtype=np.uint8)
                if m_bruta.shape[:2] != (alt_orig, larg_orig):
                    m_bruta = cv2.resize(
                        m_bruta, (larg_orig, alt_orig), interpolation=cv2.INTER_NEAREST
                    )
                m_bin = (m_bruta > 0).astype(np.uint8) * 255
                iou_atual = calcular_iou_mascaras_binarias(m_bin, mascara_gt)
                if iou_atual >= iou_melhor:
                    iou_melhor = iou_atual
                    mascara_pred_escolhida = m_bin
                    conf_escolhida = (
                        float(deteccoes.confidence[idx_d])
                        if deteccoes.confidence is not None
                        else 0.0
                    )

        if mascara_pred_escolhida is not None:
            cantos_pred_orig = extrair_4_cantos_de_mascara_ou_contorno(mascara_pred_escolhida)
            if cantos_pred_orig is not None and cantos_gt_res is not None:
                cantos_pred_res = cantos_pred_orig.copy()
                cantos_pred_res[:, 0] *= escala_x
                cantos_pred_res[:, 1] *= escala_y
                dists = np.linalg.norm(cantos_pred_res - cantos_gt_res, axis=1)
                erro_cantos_img = float(np.mean(dists))

        ious.append(iou_melhor)
        erros_cantos_px.append(erro_cantos_img)
        registros_por_imagem.append({
            "arquivo": info_img["file_name"],
            "caminho": caminho_img,
            "iou_mascara": round(iou_melhor, 4),
            "erro_cantos_px": round(erro_cantos_img, 2),
            "confianca": round(conf_escolhida, 4),
            "cantos_pred_orig": cantos_pred_orig.tolist() if cantos_pred_orig is not None else None,
        })

    if not ious:
        return {
            "score_fase2": 0.0,
            "iou_medio_mascara": 0.0,
            "map50_95_mascara": 0.0,
            "taxa_cantos_precisos_10px": 0.0,
            "erro_medio_cantos_px": 999.0,
            "erro_mediano_cantos_px": 999.0,
            "erros_individuais_px": [],
            "registros_por_imagem": [],
        }

    ious_arr = np.asarray(ious, dtype=np.float32)
    erros_arr = np.asarray(erros_cantos_px, dtype=np.float32)

    iou_medio = float(np.mean(ious_arr))
    map50_95 = float(np.mean([np.mean(ious_arr >= lim) for lim in limiares_map]))
    taxa_precisos_10px = float(np.mean(erros_arr < 10.0))
    erro_medio_px = float(np.mean(erros_arr))
    erro_mediano_px = float(np.median(erros_arr))

    score_fase2 = (
        0.45 * iou_medio
        + 0.35 * map50_95
        + 0.20 * taxa_precisos_10px
    )

    return {
        "score_fase2": round(float(score_fase2), 4),
        "iou_medio_mascara": round(iou_medio, 4),
        "map50_95_mascara": round(map50_95, 4),
        "taxa_cantos_precisos_10px": round(taxa_precisos_10px, 4),
        "taxa_cantos_precisos_10px_pct": round(taxa_precisos_10px * 100.0, 2),
        "erro_medio_cantos_px": round(erro_medio_px, 2),
        "erro_mediano_cantos_px": round(erro_mediano_px, 2),
        "erros_individuais_px": [round(float(v), 2) for v in erros_arr],
        "registros_por_imagem": registros_por_imagem,
    }

In [ ]:
# Referência do Baseline D0 (RFDETRSegSmall @ 648x648, ~69.18 MB em ONNX FP16 / ~130.5 MB em PTH)
BASELINE_DETECTOR_D0 = {
    "variante": "RFDETRSegSmall",
    "resolucao": 648,
    "tamanho_onnx_mb": 69.18,
    "iou_medio_mascara": 0.9801,
    "map50_95_mascara": 0.9223,
    "erro_medio_cantos_px": 3.13,
    "taxa_cantos_precisos_10px": 1.00,
    "score_fase2": round(0.45 * 0.9801 + 0.35 * 0.9223 + 0.20 * 1.00, 4),
}

CAMINHO_DB_OPTUNA_DRIVE = f"sqlite:///{PASTA_SAIDA_DRIVE}/estudo_optuna_fase2.db"
CAMINHO_MELHOR_PTH_DRIVE = os.path.join(PASTA_SAIDA_DRIVE, "melhor_detector_fase2.pth")
CAMINHO_HISTORICO_JSON_DRIVE = os.path.join(PASTA_SAIDA_DRIVE, "historico_trials_fase2.json")
META_TAMANHO_ONNX_MB = 45.0
N_TRIALS_FASE2 = 15
EPOCAS_POR_TRIAL_FASE2 = 18

ESTIMATIVA_TAMANHO_ONNX_FP32_MB = {
    "RFDETRSegNano": 33.5,
    "RFDETRSegSmall": 42.8,
    "RFDETRSegMedium": 54.0,
}


def estimar_tamanho_onnx_do_checkpoint_mb(caminho_pth, nome_variante):
    if not os.path.isfile(caminho_pth):
        return ESTIMATIVA_TAMANHO_ONNX_FP32_MB.get(nome_variante, 42.0)
    pacote = torch.load(caminho_pth, map_location="cpu", weights_only=False)
    pesos = pacote.get("model", pacote.get("state_dict", pacote))
    if isinstance(pesos, dict):
        total_params = sum(v.numel() for v in pesos.values() if isinstance(v, torch.Tensor))
        if total_params > 0:
            return (total_params * 4.0) / (1024.0 * 1024.0)
    return ESTIMATIVA_TAMANHO_ONNX_FP32_MB.get(nome_variante, 42.0)


def objetivo_optuna_fase2(trial):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    variantes_disponiveis = list(MAPA_CLASSES_RFDETR.keys())
    variante = trial.suggest_categorical("variante_detector", variantes_disponiveis)
    resolucao_multipla_56 = trial.suggest_categorical("resolucao_entrada", [560, 616])
    taxa_aprendizado = trial.suggest_float("taxa_aprendizado", 5e-5, 3e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-4, 1e-2, log=True)
    grad_accum_steps = trial.suggest_categorical("grad_accum_steps", [2, 4])

    # Calibra batch_size para caber com folga na VRAM da GPU T4 (15 GB) ou L4 (22.5 GB)
    tamanho_lote = 4 if grad_accum_steps == 4 else 8
    if variante == "RFDETRSegMedium" or resolucao_multipla_56 >= 616:
        tamanho_lote = 4

    # Regenera o conjunto train no SSD local com augmentation geométrico validado
    regenerar_split_treino_aumentado_ssd(
        PASTA_DATASET_COCO_BASE,
        PASTA_DATASET_COCO,
        semente_epoca=SEMENTE_GLOBAL + trial.number,
    )

    pasta_saida_trial_local = f"/content/saida_trial_{trial.number:02d}"
    if os.path.exists(pasta_saida_trial_local):
        shutil.rmtree(pasta_saida_trial_local)
    os.makedirs(pasta_saida_trial_local, exist_ok=True)

    modelo, resolucao_real = instanciar_detector_rfdetr(
        variante, resolucao_multipla_56
    )
    trial.set_user_attr("resolucao_efetiva", int(resolucao_real))

    print(
        f"\n>>> [Trial {trial.number:02d}] Variante={variante} | "
        f"Resolução={resolucao_multipla_56} (efetiva={resolucao_real}) | "
        f"LR={taxa_aprendizado:.2e} | WD={weight_decay:.2e} | "
        f"BS={tamanho_lote} x GradAccum={grad_accum_steps}"
    )

    modelo.train(
        dataset_dir=PASTA_DATASET_COCO,
        epochs=EPOCAS_POR_TRIAL_FASE2,
        batch_size=tamanho_lote,
        grad_accum_steps=grad_accum_steps,
        lr=taxa_aprendizado,
        weight_decay=weight_decay,
        warmup_epochs=2.0,
        early_stopping=True,
        early_stopping_patience=5,
        early_stopping_use_ema=True,
        tensorboard=False,
        wandb=False,
        mlflow=False,
        output_dir=pasta_saida_trial_local,
    )

    # Localiza o melhor checkpoint gerado pelo trial
    candidatos_ckpt = [
        os.path.join(pasta_saida_trial_local, "checkpoint_best_ema.pth"),
        os.path.join(pasta_saida_trial_local, "checkpoint_best_total.pth"),
        os.path.join(pasta_saida_trial_local, "checkpoint_best_regular.pth"),
        os.path.join(pasta_saida_trial_local, "checkpoint.pth"),
    ]
    caminho_ckpt_trial = None
    for cand in candidatos_ckpt:
        if os.path.isfile(cand):
            caminho_ckpt_trial = cand
            break

    if caminho_ckpt_trial is not None:
        modelo_aval, _ = instanciar_detector_rfdetr(
            variante, resolucao_real, caminho_pesos=caminho_ckpt_trial
        )
        tamanho_est_mb = estimar_tamanho_onnx_do_checkpoint_mb(caminho_ckpt_trial, variante)
    else:
        modelo_aval = modelo
        tamanho_est_mb = ESTIMATIVA_TAMANHO_ONNX_FP32_MB.get(variante, 42.0)

    pasta_val_ssd = os.path.join(PASTA_DATASET_COCO_BASE, "valid")
    metricas_val = avaliar_geometria_detector_no_split(
        modelo_aval, pasta_val_ssd, resolucao_inferencia=resolucao_real
    )
    score_fase2 = float(metricas_val["score_fase2"])

    trial.set_user_attr("tamanho_onnx_estimado_mb", round(tamanho_est_mb, 2))
    trial.set_user_attr("iou_medio_mascara", metricas_val["iou_medio_mascara"])
    trial.set_user_attr("map50_95_mascara", metricas_val["map50_95_mascara"])
    trial.set_user_attr("taxa_cantos_precisos_10px", metricas_val["taxa_cantos_precisos_10px"])
    trial.set_user_attr("erro_medio_cantos_px", metricas_val["erro_medio_cantos_px"])

    # Persiste imediatamente no Google Drive se for o melhor trial até o momento
    melhor_anterior = -1.0
    for t_ant in trial.study.get_trials(deepcopy=False):
        if t_ant.number != trial.number and t_ant.value is not None:
            if t_ant.value > melhor_anterior:
                melhor_anterior = t_ant.value

    if score_fase2 > melhor_anterior and caminho_ckpt_trial is not None:
        shutil.copy2(caminho_ckpt_trial, CAMINHO_MELHOR_PTH_DRIVE)
        metadados_vencedor = {
            "trial_vencedor": int(trial.number),
            "variante_detector": variante,
            "resolucao_solicitada": int(resolucao_multipla_56),
            "resolucao_efetiva": int(resolucao_real),
            "taxa_aprendizado": float(taxa_aprendizado),
            "weight_decay": float(weight_decay),
            "grad_accum_steps": int(grad_accum_steps),
            "tamanho_onnx_estimado_mb": round(tamanho_est_mb, 2),
            "metricas_validacao": metricas_val,
        }
        with open(
            os.path.join(PASTA_SAIDA_DRIVE, "metadados_melhor_detector_fase2.json"),
            "w",
            encoding="utf-8",
        ) as arq_m:
            json.dump(metadados_vencedor, arq_m, ensure_ascii=False, indent=2)
        print(
            f"*** [Trial {trial.number:02d}] NOVO MELHOR DETECTOR SALVO NO DRIVE: "
            f"Score={score_fase2:.4f} | IoU={metricas_val['iou_medio_mascara']:.4f} | "
            f"mAP50-95={metricas_val['map50_95_mascara']:.4f} | "
            f"ErroCantos={metricas_val['erro_medio_cantos_px']:.2f} px ***"
        )

    # Limpa pasta temporária do trial no SSD para não esgotar o disco local
    del modelo
    del modelo_aval
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if os.path.exists(pasta_saida_trial_local):
        shutil.rmtree(pasta_saida_trial_local)

    return score_fase2

In [ ]:
amostrador_fase2 = optuna.samplers.TPESampler(seed=SEMENTE_GLOBAL, multivariate=True)

ESTUDO_OPTUNA_FASE2 = optuna.create_study(
    study_name="estudo_detector_rfdetr_fase2",
    storage=CAMINHO_DB_OPTUNA_DRIVE,
    direction="maximize",
    sampler=amostrador_fase2,
    load_if_exists=True,
)

trials_ja_concluidos = len([
    t for t in ESTUDO_OPTUNA_FASE2.trials
    if t.state == optuna.trial.TrialState.COMPLETE
])

if trials_ja_concluidos == 0:
    # Prioriza a variante compacta RFDETRSegNano no primeiro trial para atingir <= 45 MB
    variante_inicial = "RFDETRSegNano" if "RFDETRSegNano" in MAPA_CLASSES_RFDETR else "RFDETRSegSmall"
    ESTUDO_OPTUNA_FASE2.enqueue_trial({
        "variante_detector": variante_inicial,
        "resolucao_entrada": 560,
        "taxa_aprendizado": 1.2e-4,
        "weight_decay": 1e-4,
        "grad_accum_steps": 2,
    })

trials_restantes = max(0, N_TRIALS_FASE2 - trials_ja_concluidos)
print(f"Banco SQLite Optuna no Drive: {CAMINHO_DB_OPTUNA_DRIVE}")
print(f"Trials já concluídos no Drive: {trials_ja_concluidos}/{N_TRIALS_FASE2} | Restantes: {trials_restantes}")

if trials_restantes > 0:
    ESTUDO_OPTUNA_FASE2.optimize(objetivo_optuna_fase2, n_trials=trials_restantes)

print("\n" + "=" * 82)
print(f"MELHOR TRIAL FASE 2: #{ESTUDO_OPTUNA_FASE2.best_trial.number}")
print(f"Melhor Score_Fase2:  {ESTUDO_OPTUNA_FASE2.best_value:.4f}")
print("Melhores Hiperparâmetros:")
for k, v in ESTUDO_OPTUNA_FASE2.best_params.items():
    print(f"  - {k}: {v}")
print("=" * 82)

## 4. Auditoria de Geometria, Exportação ONNX Limpa e Relatório com Gráficos Coloridos no Drive (`PASTA_SAIDA_DRIVE`)
1. **Exportação ONNX Limpa (`melhor_detector_fase2_fp32.onnx`):** exporta o modelo vencedor usando `opset_version=17` e `dynamo=False` (ou exportador nativo validado do `rfdetr`), verificando tamanho em MB (`<= 45 MB`), ausência total de `NaN` ou `Inf` nos tensores `dets`, `labels`, `masks` em 20 imagens reais de `valid`/`test`, e paridade numérica PyTorch vs. ONNX Runtime (`erro_max < 1e-3`).
2. **Avaliação Completa nos Conjuntos `valid` e `test`.**
3. **Geração dos 4 Gráficos Coloridos em Alta Resolução (`300 dpi`) diretamente em `PASTA_SAIDA_DRIVE`:**
   - `comparativo_detector_baseline_vs_fase2.png`
   - `distribuicao_erro_cantos_px.png`
   - `amostras_predicao_4_cantos.png`
   - `fronteira_pareto_detector.png`
4. **Salvamento do Relatório Consolidado (`relatorio_fase2.json`).**

In [ ]:
CAMINHO_ONNX_FASE2_DRIVE = os.path.join(
    PASTA_SAIDA_DRIVE, "melhor_detector_fase2_fp32.onnx"
)
CAMINHO_METADADOS_VENCEDOR = os.path.join(
    PASTA_SAIDA_DRIVE, "metadados_melhor_detector_fase2.json"
)

with open(CAMINHO_METADADOS_VENCEDOR, "r", encoding="utf-8") as arq:
    INFO_VENCEDOR_FASE2 = json.load(arq)

VARIANTE_VENCEDORA = INFO_VENCEDOR_FASE2["variante_detector"]
RESOLUCAO_VENCEDORA = int(INFO_VENCEDOR_FASE2["resolucao_efetiva"])

MODELO_VENCEDOR, _ = instanciar_detector_rfdetr(
    VARIANTE_VENCEDORA,
    RESOLUCAO_VENCEDORA,
    caminho_pesos=CAMINHO_MELHOR_PTH_DRIVE,
)

METRICAS_VALID_FASE2 = avaliar_geometria_detector_no_split(
    MODELO_VENCEDOR,
    os.path.join(PASTA_DATASET_COCO_BASE, "valid"),
    resolucao_inferencia=RESOLUCAO_VENCEDORA,
)
METRICAS_TEST_FASE2 = avaliar_geometria_detector_no_split(
    MODELO_VENCEDOR,
    os.path.join(PASTA_DATASET_COCO_BASE, "test"),
    resolucao_inferencia=RESOLUCAO_VENCEDORA,
)

print("RESULTADOS GEOMÉTRICOS DO DETECTOR VENCEDOR (FASE 2):")
print(
    f"  [VALID] Score={METRICAS_VALID_FASE2['score_fase2']:.4f} | "
    f"IoU={METRICAS_VALID_FASE2['iou_medio_mascara']:.4f} | "
    f"mAP50-95={METRICAS_VALID_FASE2['map50_95_mascara']:.4f} | "
    f"Erro 4 Cantos={METRICAS_VALID_FASE2['erro_medio_cantos_px']:.2f} px | "
    f"Cantos <10px={METRICAS_VALID_FASE2['taxa_cantos_precisos_10px_pct']:.2f}%"
)
print(
    f"  [TEST]  Score={METRICAS_TEST_FASE2['score_fase2']:.4f} | "
    f"IoU={METRICAS_TEST_FASE2['iou_medio_mascara']:.4f} | "
    f"mAP50-95={METRICAS_TEST_FASE2['map50_95_mascara']:.4f} | "
    f"Erro 4 Cantos={METRICAS_TEST_FASE2['erro_medio_cantos_px']:.2f} px | "
    f"Cantos <10px={METRICAS_TEST_FASE2['taxa_cantos_precisos_10px_pct']:.2f}%"
)

In [ ]:
def exportar_e_auditar_onnx_detector_fase2(
    modelo_rfdetr,
    resolucao,
    pasta_saida_drive,
    nome_arquivo_onnx="melhor_detector_fase2_fp32.onnx",
):
    """
    Exporta o detector RF-DETR-Seg para ONNX FP32 (opset 17, dynamo=False) e verifica:
      1. Tamanho final em MB no Google Drive;
      2. Ausência total de NaN ou Inf nos tensores de saída (dets, labels, masks) em 20 imagens reais;
      3. Paridade numérica PyTorch vs. ONNX Runtime (erro_max < 1e-3).
    """
    caminho_onnx = os.path.join(pasta_saida_drive, nome_arquivo_onnx)

    # 1. Exportação via exportador nativo do rfdetr ou torch.onnx.export (opset 17, dynamo=False)
    try:
        modelo_rfdetr.export(
            output_dir=pasta_saida_drive,
            format="onnx",
            opset_version=17,
            shape=(resolucao, resolucao),
            batch_size=1,
            fp16=False,
            verbose=False,
            output_name=nome_arquivo_onnx,
        )
    except Exception:
        rede_interna = modelo_rfdetr.model.model.cpu().eval()
        if hasattr(rede_interna, "export"):
            rede_interna.export()
        entrada_ficticia = torch.randn(1, 3, resolucao, resolucao, dtype=torch.float32)
        with torch.no_grad():
            torch.onnx.export(
                rede_interna,
                entrada_ficticia,
                caminho_onnx,
                export_params=True,
                opset_version=17,
                do_constant_folding=True,
                input_names=["input"],
                output_names=["dets", "labels", "masks"],
                dynamo=False,
            )

    grafo_onnx = onnx.load(caminho_onnx)
    onnx.checker.check_model(grafo_onnx)
    tamanho_onnx_mb = os.path.getsize(caminho_onnx) / (1024.0 * 1024.0)

    sessao_ort = ort.InferenceSession(caminho_onnx, providers=["CPUExecutionProvider"])
    nome_entrada = sessao_ort.get_inputs()[0].name
    nomes_saidas = [s.name for s in sessao_ort.get_outputs()]

    # Coleta até 20 imagens reais de valid e test para auditoria de NaN/Inf e paridade
    caminhos_teste = []
    for split in ("valid", "test", "train"):
        pasta_s = os.path.join(PASTA_DATASET_COCO_BASE, split)
        for ext in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
            caminhos_teste.extend(sorted(glob.glob(os.path.join(pasta_s, ext))))
    caminhos_20 = caminhos_teste[:20]

    media_norm = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 3, 1, 1)
    desvio_norm = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)

    rede_pt = modelo_rfdetr.model.model.cpu().eval()
    if hasattr(rede_pt, "export"):
        rede_pt.export()

    total_nan_inf = 0
    erros_max_paridade = []

    for cam_img in caminhos_20:
        with Image.open(cam_img) as img_pil:
            img_redim = img_pil.convert("RGB").resize((resolucao, resolucao), Image.BILINEAR)
        arr_rgb = np.asarray(img_redim, dtype=np.float32) / 255.0
        tensor_np = np.transpose(arr_rgb, (2, 0, 1))[np.newaxis, ...]
        tensor_np = ((tensor_np - media_norm) / desvio_norm).astype(np.float32)

        saidas_ort = sessao_ort.run(nomes_saidas, {nome_entrada: tensor_np})
        for s_arr in saidas_ort:
            if not np.all(np.isfinite(s_arr)):
                total_nan_inf += 1

        with torch.no_grad():
            saida_pt = rede_pt(torch.from_numpy(tensor_np))
            if isinstance(saida_pt, dict):
                tensores_pt = [
                    saida_pt.get("pred_boxes"),
                    saida_pt.get("pred_logits"),
                    saida_pt.get("pred_masks"),
                ]
            elif isinstance(saida_pt, (tuple, list)):
                tensores_pt = list(saida_pt)
            else:
                tensores_pt = []

            for t_pt, t_ort in zip(tensores_pt, saidas_ort):
                if t_pt is not None:
                    dif = float(np.max(np.abs(t_pt.cpu().numpy() - t_ort)))
                    erros_max_paridade.append(dif)

    erro_max_global = float(max(erros_max_paridade)) if erros_max_paridade else 0.0
    reducao_mb = BASELINE_DETECTOR_D0["tamanho_onnx_mb"] - tamanho_onnx_mb
    reducao_pct = (reducao_mb / BASELINE_DETECTOR_D0["tamanho_onnx_mb"]) * 100.0

    print("=" * 82)
    print("AUDITORIA DO MODELO ONNX EXPORTADO (FASE 2)")
    print("=" * 82)
    print(f"  1. Caminho ONNX no Drive:  {caminho_onnx}")
    print(
        f"  2. Tamanho final ONNX:     {tamanho_onnx_mb:.2f} MB "
        f"(Baseline: {BASELINE_DETECTOR_D0['tamanho_onnx_mb']:.2f} MB -> Redução: {reducao_mb:.2f} MB / {reducao_pct:.1f}%)"
    )
    print(f"  3. Tensores de saída:      {nomes_saidas}")
    print(f"  4. Ocorrências NaN/Inf (20 imagens reais): {total_nan_inf} (100% finito: {total_nan_inf == 0})")
    print(f"  5. Erro máx PyTorch vs ONNX Runtime:       {erro_max_global:.6e} (meta < 1e-3: {erro_max_global < 1e-3})")
    print("=" * 82)

    return {
        "caminho_onnx": caminho_onnx,
        "tamanho_onnx_mb": round(tamanho_onnx_mb, 3),
        "reducao_em_relacao_baseline_mb": round(reducao_mb, 3),
        "reducao_em_relacao_baseline_pct": round(reducao_pct, 2),
        "abaixo_meta_45mb": bool(tamanho_onnx_mb <= META_TAMANHO_ONNX_MB),
        "tensores_saida": nomes_saidas,
        "imagens_auditadas_nan_inf": len(caminhos_20),
        "ocorrencias_nan_ou_inf": int(total_nan_inf),
        "erro_max_paridade_pytorch_onnx": erro_max_global,
        "paridade_aprovada": bool(erro_max_global < 1e-3 and total_nan_inf == 0),
    }


AUDITORIA_ONNX_FASE2 = exportar_e_auditar_onnx_detector_fase2(
    MODELO_VENCEDOR,
    RESOLUCAO_VENCEDORA,
    PASTA_SAIDA_DRIVE,
)

In [ ]:
# 4.1 Gráfico 1: comparativo_detector_baseline_vs_fase2.png (Barras Agrupadas Coloridas)
def plotar_comparativo_detector_baseline_vs_fase2(baseline, val_fase2, audit_onnx, caminho_saida):
    sns.set_theme(style="whitegrid")
    fig, eixos = plt.subplots(1, 2, figsize=(13.5, 5.8), dpi=300)
    fig.suptitle(
        "Comparativo Detector de Tabuleiro: Baseline D0 vs. Melhor Detector Fase 2",
        fontsize=14,
        fontweight="bold",
        y=0.98,
    )

    # Painel Esquerdo: Métricas de Qualidade (%)
    rotulos_q = ["IoU Médio\nMáscara (%)", "mAP50-95\nMáscara (%)", "Cantos Precisos\n(< 10 px) (%)"]
    vals_b_q = [
        baseline["iou_medio_mascara"] * 100.0,
        baseline["map50_95_mascara"] * 100.0,
        baseline["taxa_cantos_precisos_10px"] * 100.0,
    ]
    vals_f_q = [
        val_fase2["iou_medio_mascara"] * 100.0,
        val_fase2["map50_95_mascara"] * 100.0,
        val_fase2["taxa_cantos_precisos_10px_pct"],
    ]
    x_q = np.arange(len(rotulos_q))
    larg = 0.34

    b1 = eixos[0].bar(
        x_q - larg / 2, vals_b_q, larg,
        label=f"Baseline D0 ({baseline['variante']} @ {baseline['resolucao']})",
        color="#5c6bc0", edgecolor="#1a237e", linewidth=1.1,
    )
    b2 = eixos[0].bar(
        x_q + larg / 2, vals_f_q, larg,
        label=f"Fase 2 ({VARIANTE_VENCEDORA} @ {RESOLUCAO_VENCEDORA})",
        color="#00a896", edgecolor="#004b40", linewidth=1.1,
    )
    eixos[0].set_title("Qualidade de Segmentação e 4 Cantos (Maior = Melhor)", fontsize=11.5, fontweight="bold")
    eixos[0].set_xticks(x_q)
    eixos[0].set_xticklabels(rotulos_q, fontweight="bold")
    eixos[0].set_ylim(0, 114)
    eixos[0].set_ylabel("Percentual (%)", fontweight="bold")
    eixos[0].legend(loc="lower right", frameon=True)

    for barras, cor_txt in ((b1, "#1a237e"), (b2, "#004b40")):
        for b in barras:
            h = b.get_height()
            eixos[0].annotate(
                f"{h:.2f}%",
                xy=(b.get_x() + b.get_width() / 2, h),
                xytext=(0, 4),
                textcoords="offset points",
                ha="center", va="bottom", fontsize=9.2, fontweight="bold", color=cor_txt,
            )

    # Painel Direito: Erro Médio dos 4 Cantos (px) e Tamanho ONNX (MB)
    rotulos_c = ["Erro Médio 4 Cantos\n(pixels)", "Tamanho do Modelo\nONNX (MB)"]
    vals_b_c = [baseline["erro_medio_cantos_px"], baseline["tamanho_onnx_mb"]]
    vals_f_c = [val_fase2["erro_medio_cantos_px"], audit_onnx["tamanho_onnx_mb"]]
    x_c = np.arange(len(rotulos_c))

    b3 = eixos[1].bar(
        x_c - larg / 2, vals_b_c, larg,
        label="Baseline D0",
        color="#ef5350", edgecolor="#b71c1c", linewidth=1.1,
    )
    b4 = eixos[1].bar(
        x_c + larg / 2, vals_f_c, larg,
        label="Melhor Fase 2",
        color="#ffa726", edgecolor="#e65100", linewidth=1.1,
    )
    eixos[1].set_title("Erro Geométrico (px) e Tamanho em Disco (MB) (Menor = Melhor)", fontsize=11.5, fontweight="bold")
    eixos[1].set_xticks(x_c)
    eixos[1].set_xticklabels(rotulos_c, fontweight="bold")
    eixos[1].set_ylim(0, max(vals_b_c + vals_f_c) * 1.22)
    eixos[1].set_ylabel("Pixels / Megabytes (MB)", fontweight="bold")
    eixos[1].legend(loc="upper left", frameon=True)

    unidades = ["px", "MB"]
    for barras, cor_txt in ((b3, "#b71c1c"), (b4, "#e65100")):
        for idx_u, b in enumerate(barras):
            h = b.get_height()
            eixos[1].annotate(
                f"{h:.2f} {unidades[idx_u]}",
                xy=(b.get_x() + b.get_width() / 2, h),
                xytext=(0, 4),
                textcoords="offset points",
                ha="center", va="bottom", fontsize=9.5, fontweight="bold", color=cor_txt,
            )

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(caminho_saida, dpi=300)
    plt.show()


CAMINHO_GRAFICO_COMP_DET = os.path.join(
    PASTA_SAIDA_DRIVE, "comparativo_detector_baseline_vs_fase2.png"
)
plotar_comparativo_detector_baseline_vs_fase2(
    BASELINE_DETECTOR_D0,
    METRICAS_VALID_FASE2,
    AUDITORIA_ONNX_FASE2,
    CAMINHO_GRAFICO_COMP_DET,
)

In [ ]:
# 4.2 Gráfico 2: distribuicao_erro_cantos_px.png (Histograma + Boxplot Coloridos Valid e Test)
def plotar_distribuicao_erro_cantos(metricas_valid, metricas_test, caminho_saida):
    erros_val = metricas_valid["erros_individuais_px"]
    erros_tst = metricas_test["erros_individuais_px"]

    sns.set_theme(style="whitegrid")
    fig, eixos = plt.subplots(1, 2, figsize=(13.2, 5.4), dpi=300)
    fig.suptitle(
        "Distribuição do Erro Euclidiano dos 4 Cantos (TL, TR, BR, BL) — Conjuntos Valid e Test",
        fontsize=13.5,
        fontweight="bold",
        y=0.98,
    )

    # Histograma colorido sobreposto
    sns.histplot(
        erros_val, bins=10, kde=True, color="#1e88e5",
        label=f"Validação (média={metricas_valid['erro_medio_cantos_px']:.2f} px)",
        ax=eixos[0], alpha=0.55,
    )
    sns.histplot(
        erros_tst, bins=10, kde=True, color="#fb8c00",
        label=f"Teste (média={metricas_test['erro_medio_cantos_px']:.2f} px)",
        ax=eixos[0], alpha=0.55,
    )
    eixos[0].axvline(10.0, color="#d32f2f", linestyle="--", linewidth=2.0, label="Limiar de Precisão (10 px)")
    eixos[0].set_title("Histograma e Densidade do Erro Médio por Tabuleiro", fontsize=11.5, fontweight="bold")
    eixos[0].set_xlabel("Erro Médio Euclidiano dos 4 Cantos (pixels)", fontweight="bold")
    eixos[0].set_ylabel("Quantidade de Tabuleiros", fontweight="bold")
    eixos[0].legend(frameon=True)

    # Boxplot + Stripplot colorido
    dados_box = [erros_val, erros_tst]
    rotulos_box = ["Validação (valid)", "Teste (test)"]
    paleta_box = ["#42a5f5", "#ffa726"]

    bplot = eixos[1].boxplot(
        dados_box,
        labels=rotulos_box,
        patch_artist=True,
        widths=0.42,
        medianprops={"color": "#b71c1c", "linewidth": 2.2},
    )
    for patch, cor in zip(bplot["boxes"], paleta_box):
        patch.set_facecolor(cor)
        patch.set_alpha(0.75)

    for idx_g, lista_e in enumerate(dados_box, start=1):
        jitter = np.random.normal(loc=idx_g, scale=0.05, size=len(lista_e))
        eixos[1].scatter(jitter, lista_e, color="#263238", s=36, alpha=0.85, zorder=4)

    eixos[1].axhline(10.0, color="#d32f2f", linestyle="--", linewidth=1.8, label="Teto 10 px")
    eixos[1].set_title("Boxplot e Dispersão Individual por Conjunto", fontsize=11.5, fontweight="bold")
    eixos[1].set_ylabel("Erro Euclidiano dos 4 Cantos (pixels)", fontweight="bold")
    eixos[1].legend(loc="upper right", frameon=True)

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(caminho_saida, dpi=300)
    plt.show()


CAMINHO_GRAFICO_DIST_CANTOS = os.path.join(
    PASTA_SAIDA_DRIVE, "distribuicao_erro_cantos_px.png"
)
plotar_distribuicao_erro_cantos(
    METRICAS_VALID_FASE2, METRICAS_TEST_FASE2, CAMINHO_GRAFICO_DIST_CANTOS
)

In [ ]:
# 4.3 Gráfico 3: amostras_predicao_4_cantos.png (Grade Visual 3x3 com Máscara e 4 Cantos Coloridos)
def plotar_grade_3x3_predicao_4_cantos(modelo_rfdetr, metricas_valid, metricas_test, caminho_saida):
    registros = metricas_valid["registros_por_imagem"] + metricas_test["registros_por_imagem"]
    selecionados = registros[:9]

    # Cores vibrantes e distintas em RGB para cada um dos 4 cantos
    cores_cantos_rgb = [
        (255, 59, 48),   # 1: TL (Vermelho vibrante)
        (52, 199, 89),   # 2: TR (Verde esmeralda)
        (0, 122, 255),   # 3: BR (Azul intenso)
        (255, 204, 0),   # 4: BL (Amarelo ouro)
    ]
    rotulos_cantos = ["1: TL", "2: TR", "3: BR", "4: BL"]

    fig, eixos = plt.subplots(3, 3, figsize=(15, 14.5), dpi=300)
    fig.suptitle(
        "Auditoria Visual 3x3 (Valid/Test): Máscara Segmentada e 4 Cantos (1:TL, 2:TR, 3:BR, 4:BL)",
        fontsize=14.5,
        fontweight="bold",
        y=0.98,
    )

    eixos_planos = eixos.flatten()
    for idx, ax in enumerate(eixos_planos):
        if idx >= len(selecionados):
            ax.axis("off")
            continue

        reg = selecionados[idx]
        img_bgr = cv2.imread(reg["caminho"])
        if img_bgr is None:
            ax.axis("off")
            continue

        alt, larg = img_bgr.shape[:2]
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        det = modelo_rfdetr.predict(Image.fromarray(img_rgb), threshold=0.35)

        vis = img_rgb.copy()
        if det is not None and len(det) > 0 and det.mask is not None:
            m_bruta = np.asarray(det.mask[0], dtype=np.uint8)
            if m_bruta.shape[:2] != (alt, larg):
                m_bruta = cv2.resize(m_bruta, (larg, alt), interpolation=cv2.INTER_NEAREST)
            regiao = m_bruta > 0
            cor_mascara = np.array([0, 210, 190], dtype=np.float32)
            vis[regiao] = (vis[regiao].astype(np.float32) * 0.62 + cor_mascara * 0.38).astype(np.uint8)

            cantos = extrair_4_cantos_de_mascara_ou_contorno(regiao.astype(np.uint8) * 255)
            if cantos is not None:
                pts_poly = np.round(cantos).astype(np.int32)
                cv2.polylines(vis, [pts_poly], True, (255, 255, 255), 3, cv2.LINE_AA)
                for ic, (cx, cy) in enumerate(cantos):
                    px, py = int(round(cx)), int(round(cy))
                    cor_c = cores_cantos_rgb[ic]
                    cv2.circle(vis, (px, py), 11, (255, 255, 255), -1, cv2.LINE_AA)
                    cv2.circle(vis, (px, py), 8, cor_c, -1, cv2.LINE_AA)
                    tx = max(8, min(larg - 78, px + 12))
                    ty = max(24, min(alt - 10, py - 10))
                    cv2.rectangle(vis, (tx - 4, ty - 18), (tx + 62, ty + 5), (20, 20, 25), -1)
                    cv2.putText(
                        vis, rotulos_cantos[ic], (tx, ty),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.52, cor_c, 2, cv2.LINE_AA,
                    )

        ax.imshow(vis)
        nome_curto = reg["arquivo"].split(".rf.")[0][:22]
        ax.set_title(
            f"{nome_curto} | IoU={reg['iou_mascara']:.3f} | Erro={reg['erro_cantos_px']:.2f}px",
            fontsize=10,
            fontweight="bold",
        )
        ax.axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(caminho_saida, dpi=300)
    plt.show()


CAMINHO_GRAFICO_AMOSTRAS_4CANTOS = os.path.join(
    PASTA_SAIDA_DRIVE, "amostras_predicao_4_cantos.png"
)
plotar_grade_3x3_predicao_4_cantos(
    MODELO_VENCEDOR,
    METRICAS_VALID_FASE2,
    METRICAS_TEST_FASE2,
    CAMINHO_GRAFICO_AMOSTRAS_4CANTOS,
)

In [ ]:
# 4.4 Gráfico 4: fronteira_pareto_detector.png (Dispersão Colorida Score Geométrico vs Tamanho MB)
def extrair_historico_trials_optuna(estudo):
    registros = []
    for t in estudo.trials:
        if t.state != optuna.trial.TrialState.COMPLETE or t.value is None:
            continue
        var = t.params.get("variante_detector", "RFDETRSegSmall")
        tam_mb = float(
            t.user_attrs.get(
                "tamanho_onnx_estimado_mb",
                ESTIMATIVA_TAMANHO_ONNX_FP32_MB.get(var, 42.0),
            )
        )
        registros.append({
            "trial": int(t.number),
            "variante_detector": var,
            "resolucao_entrada": int(t.params.get("resolucao_entrada", 560)),
            "resolucao_efetiva": int(t.user_attrs.get("resolucao_efetiva", 560)),
            "taxa_aprendizado": float(t.params.get("taxa_aprendizado", 1e-4)),
            "weight_decay": float(t.params.get("weight_decay", 1e-4)),
            "grad_accum_steps": int(t.params.get("grad_accum_steps", 2)),
            "tamanho_onnx_mb": round(tam_mb, 2),
            "score_fase2": round(float(t.value), 4),
            "iou_medio_mascara": float(t.user_attrs.get("iou_medio_mascara", 0.0)),
            "map50_95_mascara": float(t.user_attrs.get("map50_95_mascara", 0.0)),
            "erro_medio_cantos_px": float(t.user_attrs.get("erro_medio_cantos_px", 0.0)),
        })
    return registros


def plotar_fronteira_pareto_detector(historico_trials, baseline, caminho_saida):
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(10.5, 6.2), dpi=300)

    cores_variantes = {
        "RFDETRSegNano": "#00a896",
        "RFDETRSegSmall": "#1e88e5",
        "RFDETRSegMedium": "#8e24aa",
    }

    for var, cor in cores_variantes.items():
        sub = [r for r in historico_trials if r["variante_detector"] == var]
        if not sub:
            continue
        xs = [r["tamanho_onnx_mb"] for r in sub]
        ys = [r["score_fase2"] * 100.0 for r in sub]
        ax.scatter(
            xs, ys, label=f"{var} ({len(sub)} trials)",
            color=cor, s=110, alpha=0.88, edgecolors="black", linewidth=1.0, zorder=4,
        )
        for r in sub:
            ax.annotate(
                f"T{r['trial']} ({r['resolucao_entrada']})",
                xy=(r["tamanho_onnx_mb"], r["score_fase2"] * 100.0),
                xytext=(5, 4),
                textcoords="offset points",
                fontsize=8.2,
                fontweight="bold",
            )

    # Ponto do Baseline D0
    ax.scatter(
        [baseline["tamanho_onnx_mb"]],
        [baseline["score_fase2"] * 100.0],
        color="#e53935",
        marker="D",
        s=140,
        edgecolors="black",
        linewidth=1.2,
        label=f"Baseline D0 ({baseline['tamanho_onnx_mb']:.1f} MB)",
        zorder=5,
    )

    # Fronteira de Pareto
    ordenados = sorted(historico_trials, key=lambda r: (r["tamanho_onnx_mb"], -r["score_fase2"]))
    fronteira = []
    melhor_s = -1.0
    for item in ordenados:
        if item["score_fase2"] > melhor_s:
            fronteira.append(item)
            melhor_s = item["score_fase2"]
    if fronteira:
        ax.plot(
            [r["tamanho_onnx_mb"] for r in fronteira],
            [r["score_fase2"] * 100.0 for r in fronteira],
            linestyle="--",
            color="#ff6f00",
            linewidth=2.2,
            label="Fronteira de Pareto",
        )

    ax.axvline(
        META_TAMANHO_ONNX_MB,
        color="#b71c1c",
        linestyle="-.",
        linewidth=1.8,
        label=f"Meta de Tamanho ONNX (<= {META_TAMANHO_ONNX_MB:.0f} MB)",
    )

    ax.set_title(
        "Fronteira de Pareto do Detector (Fase 2): Score Geométrico vs. Tamanho em MB",
        fontsize=13,
        fontweight="bold",
        pad=12,
    )
    ax.set_xlabel("Tamanho do Modelo ONNX (MB)", fontsize=11, fontweight="bold")
    ax.set_ylabel("Score Geométrico Fase 2 (%)", fontsize=11, fontweight="bold")
    ax.legend(frameon=True, loc="lower right")

    plt.tight_layout()
    fig.savefig(caminho_saida, dpi=300)
    plt.show()


HISTORICO_TRIALS_FASE2 = extrair_historico_trials_optuna(ESTUDO_OPTUNA_FASE2)
CAMINHO_GRAFICO_PARETO_DET = os.path.join(
    PASTA_SAIDA_DRIVE, "fronteira_pareto_detector.png"
)
plotar_fronteira_pareto_detector(
    HISTORICO_TRIALS_FASE2, BASELINE_DETECTOR_D0, CAMINHO_GRAFICO_PARETO_DET
)

In [ ]:
# 4.5 Consolidação do Relatório Estatístico Completo no Drive (relatorio_fase2.json)
CAMINHO_RELATORIO_FASE2_DRIVE = os.path.join(PASTA_SAIDA_DRIVE, "relatorio_fase2.json")

relatorio_fase2 = {
    "fase": "Fase 2 - Fine-Tuning, Reducao de Tamanho e Estabilizacao do Detector RF-DETR-Seg",
    "formula_score_fase2": "0.45 * IoU_Medio_Mascara + 0.35 * mAP50_95_Mascara + 0.20 * Taxa_Cantos_Precisos_10px",
    "meta_tamanho_onnx_mb": META_TAMANHO_ONNX_MB,
    "baseline_d0": BASELINE_DETECTOR_D0,
    "hiperparametros_vencedores": {
        "trial_vencedor": INFO_VENCEDOR_FASE2["trial_vencedor"],
        "variante_detector": VARIANTE_VENCEDORA,
        "resolucao_solicitada": INFO_VENCEDOR_FASE2["resolucao_solicitada"],
        "resolucao_efetiva": RESOLUCAO_VENCEDORA,
        "taxa_aprendizado": INFO_VENCEDOR_FASE2["taxa_aprendizado"],
        "weight_decay": INFO_VENCEDOR_FASE2["weight_decay"],
        "grad_accum_steps": INFO_VENCEDOR_FASE2["grad_accum_steps"],
    },
    "metricas_valid": {
        k: v for k, v in METRICAS_VALID_FASE2.items() if k != "registros_por_imagem"
    },
    "metricas_test": {
        k: v for k, v in METRICAS_TEST_FASE2.items() if k != "registros_por_imagem"
    },
    "auditoria_onnx": AUDITORIA_ONNX_FASE2,
    "historico_trials": HISTORICO_TRIALS_FASE2,
    "arquivos_gerados_no_drive": {
        "banco_optuna_sqlite": CAMINHO_DB_OPTUNA_DRIVE,
        "checkpoint_pytorch_vencedor": CAMINHO_MELHOR_PTH_DRIVE,
        "modelo_onnx_fp32": CAMINHO_ONNX_FASE2_DRIVE,
        "comparativo_detector_baseline_vs_fase2": CAMINHO_GRAFICO_COMP_DET,
        "distribuicao_erro_cantos_px": CAMINHO_GRAFICO_DIST_CANTOS,
        "amostras_predicao_4_cantos": CAMINHO_GRAFICO_AMOSTRAS_4CANTOS,
        "fronteira_pareto_detector": CAMINHO_GRAFICO_PARETO_DET,
        "relatorio_fase2_json": CAMINHO_RELATORIO_FASE2_DRIVE,
    },
}

with open(CAMINHO_RELATORIO_FASE2_DRIVE, "w", encoding="utf-8") as arq:
    json.dump(relatorio_fase2, arq, ensure_ascii=False, indent=2)

print(f"Relatório completo da Fase 2 salvo no Google Drive: {CAMINHO_RELATORIO_FASE2_DRIVE}")
print("Arquivos presentes em PASTA_SAIDA_DRIVE:", sorted(os.listdir(PASTA_SAIDA_DRIVE)))

## 5. Célula Autônoma de Reexportação ONNX em CPU (Economia de Créditos GPU)
Esta célula é **totalmente independente** e pode ser executada em um ambiente **CPU gratuito** do Google Colab após o término do treino: ela monta o Google Drive, lê `melhor_detector_fase2.pth` e `metadados_melhor_detector_fase2.json` diretamente de `PASTA_SAIDA_DRIVE`, exporta `melhor_detector_fase2_fp32.onnx` (`opset_version=17`, `dynamo=False`) e valida o grafo ONNX com `onnxruntime` sem consumir créditos de GPU.

In [ ]:
def reexportar_detector_vencedor_em_cpu_autonomo():
    import json
    import os
    import sys
    import numpy as np
    import onnx
    import onnxruntime as ort
    import rfdetr
    import torch

    if "google.colab" in sys.modules:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")

    pasta_projeto_fase2 = "/content/drive/MyDrive/doutorado/20261 visao computacional/projeto/dataset3"
    pasta_saida_drive = f"{pasta_projeto_fase2}/resultados_fase2"
    caminho_pth = os.path.join(pasta_saida_drive, "melhor_detector_fase2.pth")
    caminho_meta = os.path.join(pasta_saida_drive, "metadados_melhor_detector_fase2.json")
    nome_onnx = "melhor_detector_fase2_fp32.onnx"
    caminho_onnx = os.path.join(pasta_saida_drive, nome_onnx)

    if not os.path.isfile(caminho_pth):
        raise FileNotFoundError(f"Checkpoint vencedor não encontrado no Drive: {caminho_pth}")

    variante = "RFDETRSegNano"
    resolucao = 560
    if os.path.isfile(caminho_meta):
        with open(caminho_meta, "r", encoding="utf-8") as arq:
            meta = json.load(arq)
        variante = meta.get("variante_detector", variante)
        resolucao = int(meta.get("resolucao_efetiva", resolucao))

    classe_det = getattr(rfdetr, variante)
    detector = classe_det(pretrain_weights=caminho_pth, resolution=resolucao, num_classes=1)

    try:
        detector.export(
            output_dir=pasta_saida_drive,
            format="onnx",
            opset_version=17,
            shape=(resolucao, resolucao),
            batch_size=1,
            fp16=False,
            verbose=False,
            output_name=nome_onnx,
        )
    except Exception:
        rede = detector.model.model.cpu().eval()
        if hasattr(rede, "export"):
            rede.export()
        entrada_dummy = torch.randn(1, 3, resolucao, resolucao, dtype=torch.float32)
        with torch.no_grad():
            torch.onnx.export(
                rede,
                entrada_dummy,
                caminho_onnx,
                export_params=True,
                opset_version=17,
                do_constant_folding=True,
                input_names=["input"],
                output_names=["dets", "labels", "masks"],
                dynamo=False,
            )

    grafo = onnx.load(caminho_onnx)
    onnx.checker.check_model(grafo)
    tam_mb = os.path.getsize(caminho_onnx) / (1024.0 * 1024.0)

    sessao = ort.InferenceSession(caminho_onnx, providers=["CPUExecutionProvider"])
    nome_in = sessao.get_inputs()[0].name
    nomes_out = [o.name for o in sessao.get_outputs()]
    teste_np = np.random.randn(1, 3, resolucao, resolucao).astype(np.float32)
    saidas = sessao.run(nomes_out, {nome_in: teste_np})
    finito = all(bool(np.all(np.isfinite(s))) for s in saidas)

    print(f"[CPU Reexportação Concluída] Arquivo: {caminho_onnx}")
    print(f"Variante: {variante} @ {resolucao}x{resolucao} | Tamanho: {tam_mb:.2f} MB | Saídas finitas: {finito}")


# Descomente ou execute diretamente em runtime CPU caso deseje reexportar sem usar GPU:
# reexportar_detector_vencedor_em_cpu_autonomo()